In [4]:
# %pip uninstall torch torchvision torchaudio -y

In [5]:
# %pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [6]:
import re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split

print(f"PyTorch versão {torch.__version__} importado com sucesso!")

# Definindo o dispositivo (Usa GPU se disponível, senão CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo de processamento: {device}")

PyTorch versão 2.11.0+cu126 importado com sucesso!
Dispositivo de processamento: cuda


In [7]:
# Dataset de exemplo
dados = {
'texto': [
'eu odeio esse tipo de gente, voces sao um lixo',
'hoje o dia esta muito bonito para passear',
'morte a todos os que sao diferentes de nos',
'espero que voce tenha um otimo fim de semana',
'pessoas dessa raca deveriam ser exterminadas',
'parabens pelo seu trabalho, ficou excelente',
'voce e uma pessoa horrivel e nao merece viver',
'vamos jantar naquele restaurante novo hoje?'
],
'label': [1, 0, 1, 0, 1, 0, 1, 0]
}

df = pd.DataFrame(dados)
X = df['texto'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Tamanho do treino: {len(X_train)} amostras")
print(f"Tamanho do teste: {len(X_test)} amostras")

Tamanho do treino: 6 amostras
Tamanho do teste: 2 amostras


In [8]:
MAX_LEN = 15 # Tamanho maximo da frase

# 1. Construindo o vocabulário a partir dos dados de treino
todas_as_palavras = [palavra for frase in X_train for palavra in frase.lower().split()]
frequencia_palavras = Counter(todas_as_palavras)

# Mapeando palavras para indices. Reservamos 0 para Padding e 1 para Unknown (OOV)
vocabulario = {palavra: idx + 2 for idx, (palavra, _) in enumerate(frequencia_palavras.most_common())}
vocabulario['<PAD>'] = 0
vocabulario['<UNK>'] = 1

# Removendo virgulas
def limpar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    return texto

# Função par converter texto em sequencia de numeros com padding
def texto_para_sequencia(texto, vocab, max_len):
    tokens = limpar_texto(texto).split()
    #tokens = texto.lower().split()
    seq = [vocab.get(token, vocab['<UNK>']) for token in tokens]
    
    # Padding (preenchimento com zeros) ou truncamento (corte)
    if len(seq) < max_len:
        seq = seq + [vocab['<PAD>' ]] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    return seq

# 2. Criando a classe Dataset do PyTorch
class TextoDataset(Dataset):
    def __init__(self, X, y, vocab, max_len):
        self.X = [texto_para_sequencia(texto, vocab, max_len) for texto in X]
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Retorna a sequencia como tensor de inteiros e a label como tensor float
        return torch.tensor(self.X[idx], dtype=torch. long), torch.tensor(self.y[idx], dtype=torch.float32)

# Instanciando Datasets e DataLoaders (para gerar batches)
train_dataset = TextoDataset(X_train, y_train, vocabulario, MAX_LEN)
test_dataset = TextoDataset(X_test, y_test, vocabulario, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

print("Tamanho do Vocabulário:", len(vocabulario))

Tamanho do Vocabulário: 44


## 4. Definindo a Arquitetura da Rede Neural (LSTM)

No PyTorch, criamos modelos herdando da classe ``nn. Module``. Definimos as camadas no metodo ``__init__``
fluem por essas camadas no metodo ``forward``.

In [9]:
class ModeloHateSpeechLSTM(nn.Module):
    def __init__(self, tamanho_vocab, dimensao_embedding, dimensao_oculta):
        super(ModeloHateSpeechLSTM, self).__init__()

        # Camada de Embedding: transforma inteiros em vetores densos.
        # padding_idx=0 diz ao PyTorch para ignorar os zeros usados no padding
        self.embedding = nn.Embedding(tamanho_vocab, dimensao_embedding, padding_idx=0)

        # Camada LSTM: batch_first=True indica que o formato do tensor é (batch, sequencia, caracteristicas)
        self.lstm = nn.LSTM(dimensao_embedding, dimensao_oculta, batch_first=True)
        
        # Dropout para regularização
        self.dropout = nn.Dropout(0.2)
        
        # Camada Totalmente Conectada (Dense) para classificação
        self.fc = nn.Linear(dimensao_oculta, 1)
        
        # Ativação Sigmoid para transformar a sáida final em probabilidade (0 a 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # x shape: (batch_size, max_len)
        x = self.embedding(x)
        # x shape: (batch_size, max_len, dimensao_embedding)

        lstm_out, (hidden, cell) = self.lstm(x)

        # A LSTM retorna todos os estados (lstm_out) e os estados finais (hidden, cell).
        # Como queremos classificar a frase toda, pegamos apenas o ultimo hidden state.
        ultimo_hidden_state = hidden[-1]

        out = self.dropout(ultimo_hidden_state)
        out = self.fc(out)
        out = self.sigmoid(out)

        return out.squeeze() # Remove dimensões extras (ex: de [batch_size, 1] para [batch_size])

# Instanciando o modelo e movendo para a CPU/GPU
tamanho_vocab = len(vocabulario)
dim_embedding = 16
dim_oculta = 32

modelo = ModeloHateSpeechLSTM(tamanho_vocab, dim_embedding, dim_oculta).to(device)
print(modelo)

ModeloHateSpeechLSTM(
  (embedding): Embedding(44, 16, padding_idx=0)
  (lstm): LSTM(16, 32, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


## 5. O Loop de Treinamento

No PyTorch, você deve escrever explicitamente os passos de:

1. Zerar os gradientes (``optimizer.zero_grad()``).
2. Fazer a previsão (``forward pass``).
3. Calcular o erro (``loss``).
4. Calcular os gradientes (``backward pass``).
5. Atualizar os pesos (``optimizer.step()``).

In [10]:
# Função de Custo (Binary Cross Entropy) e Otimizador (Adam)
criterio = nn. BCELoss()
otimizador = optim.Adam(modelo.parameters(), lr=0.01)

def calcular_acuracia(previsoes, labels):
    # Arredonda as previsões para 0 ou 1
    previsoes_arredondadas = torch.round(previsoes)

    # Compara quantas previsões são iguais aos rótulos reais
    corretas = (previsoes_arredondadas == labels). float()

    # Calcula a média (ex: 8 acertos em 10 = 0.8 ou 80%)
    acuracia = corretas.sum() / len(corretas)
    return acuracia

epocas = 100

for epoca in range(epocas):
    modelo.train() # Coloca o modelo em modo de treinamento (ativa dropout, etc)
    loss_treino = 0.0
    acc_treino = 0.0

    for textos, labels in train_loader:
        textos, labels = textos. to(device), labels. to(device)

        otimizador.zero_grad()
        previsoes = modelo(textos)

        loss = criterio(previsoes, labels)
        acc = calcular_acuracia(previsoes, labels)
        loss.backward()
        otimizador.step()

        loss_treino += loss.item()
        acc_treino += acc.item()


loss_media = loss_treino / len(train_loader)
acc_media = acc_treino / len(train_loader)

if (epoca+1) % 5 == 0 or epoca == 0:
    print(f"Época: {epoca+1}/{epocas} | Los: {loss_media:.4f} | Acurácia: {acc_media*100:.2f}%")

Época: 100/100 | Los: 0.0002 | Acurácia: 100.00%


In [11]:
# Coloca o modelo em modo de avaliação (desliga Dropout, etc.)
modelo.eval()

loss_teste = 0.0
acc_teste = 0.0

# torch.no_grad() diz ao PyTorch para não calcular gradientes aqui,
# economizando muita memória e processamento.
with torch.no_grad():
    for textos, labels in test_loader:
        textos, labels = textos.to(device), labels.to(device)

        previsoes = modelo(textos)
        loss = criterio(previsoes, labels)
        acc = calcular_acuracia(previsoes, labels)

        loss_teste += loss.item()
        acc_teste += acc.item()

loss_teste_media = loss_teste / len(test_loader)
acc_teste_media = acc_teste / len(test_loader)

print("-"*30)
print(f"RESULTADOS NO CONJUNTO DE TESTE:")
print(f"Loss: {loss_teste_media:.4f}")
print(f"Acurácia: {acc_teste_media*100:.2f}%")
print("-" * 30)

------------------------------
RESULTADOS NO CONJUNTO DE TESTE:
Loss: 0.0002
Acurácia: 100.00%
------------------------------


In [12]:
def detectar_odio(frase, modelo, vocab):
    modelo.eval() # Coloca o modelo em modo de avaliação (desliga dropout)

    # Preprocessamento manual da frase
    seq = texto_para_sequencia(frase, vocab, MAX_LEN)

    # Converte para tensor, adiciona dimensão de batch [1, MAX_LEN] e joga para o device
    tensor_entrada = torch.tensor([seq], dtype=torch.long).to(device)
    
    # Evita calcular gradientes durante a inferência (economiza-memoria e tempo)
    with torch.no_grad():
        predicao = modelo(tensor_entrada).item()

    if predicao > 0.5:
        resultado = "DISCURSO DE ÓDIO DETECTADO !!! "
    else:
        resultado = "TEXTO NORMAL/NEUTRO"
        
    print(f"Frase: '{frase}'")
    print(f"Predição Bruta: {predicao:.4f}")
    print(f"Classificação: {resultado}\n")
    
# Testes práticos
detectar_odio("eu acho que voces deveriam todos morrer", modelo, vocabulario)
detectar_odio("que dia excelente para estudar programacao", modelo, vocabulario)

Frase: 'eu acho que voces deveriam todos morrer'
Predição Bruta: 0.0002
Classificação: TEXTO NORMAL/NEUTRO

Frase: 'que dia excelente para estudar programacao'
Predição Bruta: 0.0002
Classificação: TEXTO NORMAL/NEUTRO

